# Lecture 3 — Class Exercise
## Line Charts: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
- No spaghetti — if you show multiple lines, grey everything and highlight one
- Remove chart borders, heavy gridlines, rotating text
- Label directly; avoid legends where possible
- Insight title


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

RAW_URL = 'https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv'
raw = pd.read_csv(RAW_URL)

regions = {
    'United States': 'North America', 'Canada': 'North America',
    'Brazil': 'Latin America', 'Mexico': 'Latin America',
    'United Kingdom': 'Europe', 'Germany': 'Europe', 'France': 'Europe', 'Russia': 'Europe',
    'China': 'Asia', 'India': 'Asia', 'Japan': 'Asia', 'South Korea': 'Asia', 'Indonesia': 'Asia',
    'Australia': 'Oceania', 'South Africa': 'Africa',
}

df = (raw[raw['country'].isin(list(regions)) & raw['year'].between(2000, 2022)]
      .rename(columns={'country': 'Country', 'year': 'Year',
                       'co2': 'CO2_Mt', 'co2_per_capita': 'CO2_per_capita'})
      [['Country', 'Year', 'CO2_Mt', 'CO2_per_capita']]
      .dropna(subset=['CO2_Mt', 'CO2_per_capita']))
df['Region'] = df['Country'].map(regions)

print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())

Loaded: 345 rows | Countries: 15 | Years: 2000-2022
        Country  Year   CO2_Mt  CO2_per_capita   Region
3654  Australia  2000  349.898          18.288  Oceania
3655  Australia  2001  357.668          18.469  Oceania
3656  Australia  2002  362.425          18.495  Oceania
3657  Australia  2003  370.784          18.708  Oceania
3658  Australia  2004  382.005          19.056  Oceania


In [ ]:
print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))

Countries: ['Australia' 'Brazil' 'Canada' 'China' 'France' 'Germany' 'India'
 'Indonesia' 'Japan' 'Mexico' 'Russia' 'South Africa' 'South Korea'
 'United Kingdom' 'United States']

CO2 range: 281.054 to 11711.808 Mt

Regional averages (2022):
Region
Asia             3387.5
North America    2801.5
Europe            737.4
Latin America     466.9
Africa            428.8
Oceania           384.1
Name: CO2_Mt, dtype: float64


## Task 1 - A line chart showing CO2 per Capita for all countries in the dataset

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#EEEEEE), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

**Story:** The United Kingdom cut its CO2 per capita by more than half between 2000 and 2022 — the steepest decline of all 15 countries. France, the US and Germany follow, while China, India and Indonesia moved the other way: China's per-capita emissions almost tripled.

In [ ]:
highlight = 'United Kingdom'

hl = df.loc[df['Country'] == highlight]
start = hl['CO2_per_capita'].iloc[0]
end = hl['CO2_per_capita'].iloc[-1]
pct_change = (end - start) / start * 100

color_map = {c: 'darkblue' if c == highlight else '#EEEEEE' for c in df['Country'].unique()}

fig = px.line(data_frame=df,
              x='Year', y='CO2_per_capita',
              color='Country',
              color_discrete_map=color_map,
              labels={'CO2_per_capita': 'CO2 per Capita (t)', 'Year': ''},
              title=f'UK CO2 per capita fell {abs(pct_change):.0f}% since 2000 — the steepest drop of all 15 countries',
              template='plotly_white',
              markers=False,
              height=450)

fig.update_traces(
    line=dict(width=1.5),
    showlegend=False
)

fig.update_traces(
    line=dict(width=3),
    selector=dict(name=highlight)
)

fig.add_annotation(
    x=2022, y=end,
    text=highlight, showarrow=False,
    xanchor='left', xshift=10,
    font=dict(color='darkblue', size=12, weight='bold', family='Arial')
)

fig.add_annotation(
    x=2022, y=df.loc[df['Year'] == 2022, 'CO2_per_capita'].max(),
    text='Other countries', showarrow=False,
    xanchor='left', xshift=10,
    font=dict(color='darkgrey', size=12, family='Arial')
)

fig.update_layout(
    font=dict(size=13),
    yaxis=dict(showgrid=False),
    xaxis=dict(showgrid=False, range=[2000, 2022]),
    margin=dict(r=130)
)

fig.show()

---
## Task 2 — A line chart showing CO2 emissions over time for all Asian countries in the dataset, with one country highlighted.

**same requirements as Task1**

**Story:** China dominates Asia's emissions, but India is the story to watch — its emissions almost tripled since 2000 (+187%), it overtook Japan in 2006 and is now Asia's second-largest emitter. Japan is the only Asian country in the dataset whose emissions fell.

In [ ]:
asia = df.loc[df['Region'] == 'Asia']
highlight = 'India'

hl = asia.loc[asia['Country'] == highlight]
start = hl['CO2_Mt'].iloc[0]
end = hl['CO2_Mt'].iloc[-1]
pct_change = (end - start) / start * 100

color_map = {c: 'crimson' if c == highlight else '#EEEEEE' for c in asia['Country'].unique()}

fig = px.line(data_frame=asia,
              x='Year', y='CO2_Mt',
              color='Country',
              color_discrete_map=color_map,
              labels={'CO2_Mt': 'CO2 Emissions (Mt)', 'Year': ''},
              title=f"India's CO2 emissions rose {pct_change:.0f}% since 2000 — overtaking Japan to become Asia's #2 emitter",
              template='plotly_white',
              markers=False,
              height=450)

fig.update_traces(
    line=dict(width=1.5),
    showlegend=False
)

fig.update_traces(
    line=dict(width=3),
    selector=dict(name=highlight)
)

fig.add_annotation(
    x=2022, y=end,
    text=highlight, showarrow=False,
    xanchor='left', xshift=10,
    font=dict(color='crimson', size=12, weight='bold', family='Arial')
)

china_label = asia.loc[(asia['Country'] == 'China') & (asia['Year'] == 2022)]
japan_label = asia.loc[(asia['Country'] == 'Japan') & (asia['Year'] == 2022)]

fig.add_annotation(
    x=2022, y=china_label['CO2_Mt'].values[0],
    text='China', showarrow=False,
    xanchor='left', xshift=10,
    font=dict(color='darkgrey', size=12, family='Arial')
)

fig.add_annotation(
    x=2022, y=japan_label['CO2_Mt'].values[0],
    text='Japan, Indonesia,<br>South Korea', showarrow=False,
    xanchor='left', xshift=10, align='left',
    font=dict(color='darkgrey', size=11, family='Arial')
)

fig.update_layout(
    font=dict(size=13),
    yaxis=dict(showgrid=False),
    xaxis=dict(showgrid=False, range=[2000, 2022]),
    margin=dict(r=130)
)

fig.show()